# Brain Tumor MRI — Model Evaluation

This notebook provides a comprehensive evaluation of the trained ResNet18 model on the **held-out test set**.

**Metrics computed:**
- Overall accuracy, precision, recall, F1-score (weighted)
- Per-class breakdown table
- Confusion matrix heatmap
- Sample prediction visualizations
- All results saved to `model_evaluation.json`

---

## Step 1: Environment Setup

In [ ]:
!pip install scikit-learn seaborn --quiet

# Mount Google Drive to access the saved model
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision import datasets, transforms, models
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

DATA_ROOT    = '/content/brain_tumor_dataset'
TEST_DIR     = os.path.join(DATA_ROOT, 'Testing')
MODEL_PATH   = '/content/drive/MyDrive/BrainTumorModel/resnet18_braintumor.pt'
if not os.path.exists(MODEL_PATH):
    MODEL_PATH = '/content/resnet18_braintumor.pt'

RESULTS_PATH = '/content/model_evaluation.json'
BATCH_SIZE   = 32
NUM_CLASSES  = 4

print(f'Model path  : {MODEL_PATH}')
print(f'Model exists: {os.path.exists(MODEL_PATH)}')

## Step 2: Load the Trained Model

Reconstruct the ResNet18 architecture and load the saved weights from the checkpoint produced in notebook 02.

In [ ]:
checkpoint  = torch.load(MODEL_PATH, map_location=device)
CLASS_NAMES = checkpoint.get('class_names', ['glioma', 'meningioma', 'no_tumor', 'pituitary'])

print(f'Classes from checkpoint : {CLASS_NAMES}')
print(f'Best val acc (training) : {checkpoint.get("best_val_acc", "N/A")}')

model = resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print('\nModel loaded and set to eval mode.')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## Step 3: Prepare Test DataLoader

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

test_dataset = datasets.ImageFolder(root=TEST_DIR, transform=test_transforms)
test_loader  = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=(device.type == 'cuda')
)

print(f'Test samples : {len(test_dataset)}')
print(f'Test batches : {len(test_loader)}')
print(f'Classes      : {test_dataset.classes}')

## Step 4: Run Inference on the Full Test Set

Collect all predictions, ground-truth labels, and softmax probabilities.

In [ ]:
all_preds  = []
all_labels = []
all_probs  = []

model.eval()
with torch.no_grad():
    for batch_idx, (imgs, labels) in enumerate(test_loader):
        imgs    = imgs.to(device)
        outputs = model(imgs)
        probs   = torch.softmax(outputs, dim=1)
        preds   = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())
        if (batch_idx + 1) % 5 == 0:
            print(f'Processed batch {batch_idx+1}/{len(test_loader)}')

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)
print(f'\nInference complete. Total predictions: {len(all_preds)}')

## Step 5: Compute Overall Metrics

Calculate accuracy, weighted precision, recall, and F1-score.

In [ ]:
accuracy  = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
recall    = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
f1        = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

print('=' * 40)
print('     OVERALL TEST SET METRICS')
print('=' * 40)
print(f'  Accuracy          : {accuracy*100:.2f}%')
print(f'  Precision (wtd)   : {precision*100:.2f}%')
print(f'  Recall (wtd)      : {recall*100:.2f}%')
print(f'  F1-Score (wtd)    : {f1*100:.2f}%')
print('=' * 40)

## Step 6: Per-Class Metrics Table

In [ ]:
report_dict = classification_report(
    all_labels, all_preds,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

print(f'  {"Class":<14} {"Precision":>10} {"Recall":>10} {"F1-Score":>10} {"Support":>9}')
print('  ' + '-' * 58)
for cls_name in CLASS_NAMES:
    r = report_dict[cls_name]
    print(f'  {cls_name:<14} {r["precision"]*100:>9.2f}%'
          f' {r["recall"]*100:>9.2f}%'
          f' {r["f1-score"]*100:>9.2f}%'
          f' {int(r["support"]):>9}')

print('\nFull sklearn classification report:')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, zero_division=0))

## Step 7: Confusion Matrix Heatmap

Diagonal entries = correct predictions. Off-diagonal = misclassifications.  
Left panel shows raw counts; right panel shows row-normalized percentages.

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Confusion Matrix — Brain Tumor MRI Test Set', fontsize=15, fontweight='bold')

sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    linewidths=0.5, ax=axes[0]
)
axes[0].set_title('Raw Counts', fontsize=13)
axes[0].set_xlabel('Predicted Label', fontsize=11)
axes[0].set_ylabel('True Label', fontsize=11)
axes[0].tick_params(axis='x', rotation=30)

sns.heatmap(
    cm_normalized, annot=True, fmt='.1f', cmap='YlOrRd',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    linewidths=0.5, ax=axes[1]
)
axes[1].set_title('Normalized (% per true class)', fontsize=13)
axes[1].set_xlabel('Predicted Label', fontsize=11)
axes[1].set_ylabel('True Label', fontsize=11)
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: /content/confusion_matrix.png')

## Step 8: Sample Predictions with Images

Display 16 random test images with their true and predicted labels.  
**Green** = correct prediction | **Red** = incorrect prediction

In [ ]:
import random

# Load without normalization so images look natural
display_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])
display_dataset = datasets.ImageFolder(root=TEST_DIR, transform=display_transforms)
normalize       = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                        std=[0.229, 0.224, 0.225])

SAMPLE_COUNT = 16
indices = random.sample(range(len(display_dataset)), SAMPLE_COUNT)

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
fig.suptitle('Sample Test Predictions\n(Green = Correct | Red = Incorrect)',
             fontsize=14, fontweight='bold')

model.eval()
for ax_idx, img_idx in enumerate(indices):
    img_tensor, true_label = display_dataset[img_idx]
    model_input = normalize(img_tensor).unsqueeze(0).to(device)

    with torch.no_grad():
        output     = model(model_input)
        pred_label = output.argmax(dim=1).item()
        confidence = torch.softmax(output, dim=1)[0, pred_label].item()

    ax = axes[ax_idx // 4][ax_idx % 4]
    ax.imshow(img_tensor.permute(1, 2, 0).numpy())
    ax.axis('off')

    is_correct = (pred_label == true_label)
    color      = 'green' if is_correct else 'red'
    ax.set_title(
        f'True: {CLASS_NAMES[true_label]}\nPred: {CLASS_NAMES[pred_label]} ({confidence*100:.1f}%)',
        fontsize=8, color=color, fontweight='bold'
    )
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(3)
        spine.set_visible(True)

plt.tight_layout()
plt.savefig('/content/sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: /content/sample_predictions.png')

## Step 9: Save All Metrics to model_evaluation.json

In [ ]:
import json, shutil
from datetime import datetime

per_class_metrics = {}
for cls_name in CLASS_NAMES:
    r = report_dict[cls_name]
    per_class_metrics[cls_name] = {
        'precision': round(r['precision'], 4),
        'recall':    round(r['recall'],    4),
        'f1_score':  round(r['f1-score'],  4),
        'support':   int(r['support'])
    }

evaluation_results = {
    'metadata': {
        'timestamp':    datetime.now().isoformat(),
        'model':        'ResNet18',
        'dataset':      'masoudnickparvar/brain-tumor-mri-dataset',
        'test_samples': int(len(all_labels)),
        'classes':      CLASS_NAMES,
    },
    'overall_metrics': {
        'accuracy':           round(float(accuracy),  4),
        'precision_weighted': round(float(precision), 4),
        'recall_weighted':    round(float(recall),    4),
        'f1_weighted':        round(float(f1),        4),
    },
    'per_class_metrics': per_class_metrics,
    'confusion_matrix':  cm.tolist(),
}

with open(RESULTS_PATH, 'w') as f:
    json.dump(evaluation_results, f, indent=2)

print(f'Saved: {RESULTS_PATH}')

drive_path = '/content/drive/MyDrive/BrainTumorModel/model_evaluation.json'
shutil.copy(RESULTS_PATH, drive_path)
print(f'Copied to Drive: {drive_path}')

print('\nOverall metrics preview:')
print(json.dumps(evaluation_results['overall_metrics'], indent=2))
print('\nPer-class metrics preview:')
for cls, m in evaluation_results['per_class_metrics'].items():
    print(f'  {cls}: {m}')

## Summary

| File | Description |
|------|-------------|
| `model_evaluation.json` | All metrics in structured JSON |
| `confusion_matrix.png` | Raw + normalized heatmaps |
| `sample_predictions.png` | 16 test images with predictions |

**Next step:** Open `04_rag_setup.ipynb` to build the Retrieval-Augmented Generation clinical guidance system.